<a href="https://colab.research.google.com/github/Alfred9/Exploring-LLMs/blob/main/RAG_Architectures/Adaptive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install -U faiss-cpu langchain langchain-core langchain-community langchain-openai langchain-text-splitters python-dotenv pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.6 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [ ]:
#load data

In [ ]:
# standard library
import os
import sys
from typing import Dict, Any


from dotenv import load_dotenv

from langchain_core.prompts import PromptTemplate

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import BaseModel, Field

# local modules
#from helper_functions import *
#from evaluation.evalute_rag import *

# load environment variables
#load_dotenv()

os.environ["OPEN_API_KEY"] = os.getenv('OPEN_API_KEY')

### Query Classifier

In [ ]:
class category_options(BaseModel):
    category: str = Field(
        description="The category of the query: Factual, Opinion, Analytical, or Contextual",
        example="Factual"
    )

class  QueryClassifier:
  def __init__(self):
    self.llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)
    self.prompt = PromptTemplate(
        input=["query"],
        template="Clasify the following query into one of these categoreies:Factual, Analytical, Opinion, or Contextual. \nQuery:{query}\nCategory:"
    )
    self.chain = self.prompt | self.llm.with_structured_output(category_options)


    def classify(self, query):
      print("clasiffying query")
      return self.chain.invoke(query).category

### Base Retriever Class

In [ ]:
class BaseRetrievalStrategy:
  def __init__(self, texts):
    self.embedding = OpenAIEmbeddings()
    text_splitter = CharacterTextSplitter(chunk=80, chunk_overlap=0)
    self.documents = text_splitter.create_documents(texts)
    self.db = FAISS.from_documents(self.documents, self.embedding)
    self.llm = ChatOpenAI(temperature=0, model="gpt-40", max_tokens=4000)

    def retrieve(self, query, k=4):
      return self.db.similarity_search(query, k=k)

### Factual Retriver Strategy

In [ ]:
class relevant_score(BaseModel):
  score:float = Field(description="The relevance score of the document to the query", example=8.0)

class FactualRetrieval(BaseRetrievalStrategy):
  def retrieve(self, query, k=4):
    print("retrieving factual")

    #we use llm to enhance the query
    enhanced_query_prompt = PromptTemplate(
        input_variables=["query"]
        template="Enhance this factual query for better information retrieval: {query}"
    )
    query_chain = enhanced_query_prompt | self.llm
    enhanced_query = query_chain.invoke(query).content
    print(f'enhandle query: {enhanced_query}')

    #document retrieval using the enhanced query
    docs = self.db.similarity_search(enhanced_query, k=k*2)

    #we use llm to rank the documents relevance
    ranking_prompt = PromptTemplate(
        input_variables=["query", "doc"]
        template="On a scale of 1-10, how relevant is this document to the query: '{query}'?\nDocument\n: {doc}\nRelevance Score:"
    )
    ranking_chain = ranking_prompt | self.llm.with_structured_output(relevant_score)

    ranked_docs = []
    print("ranking docs")
    for doc in docs:
      input_data = {"query": enhanced_query, "doc": doc.page_content}
      score = float(ranking_chain.invoke(input_data).score)
      ranked_docs.append((doc,score))

    #sort by  relevance score and return top k
    ranked_docs.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked_docs[:k]]


### Opinion Retrieval Strategy

In [ ]:
class OpinionRetrieval(BaseRetrievalStrategy):
  def retrieve(self, query, k=3):
    print("retrieving opinion")
    #LLM to identify potential viewpoints
    viewpoints_prompt = PromptTemplate(
        input_variables=["query", "k"],
        template="Identify {k} distinct viewpoints or perspective on the topic: {query}"
    )
    viewpoints_chain = viewpoints_prompt | self.llm
    input_data = {"query":query, "k":k}
    viewpoints = viewpoints_chain.invoke(input_data).content.split('\n')
    print(f'viewpoints: {viewpoints}')


    all_docs = []
    for viewpoint in viewpoints:
      all_docs.extend(self.db.similarity_search(f"{query} {viewpoint}", k=2))

    #LLM to classify and select diverse opinions
    opinion_prompt = PromptTemplate(
        input_variables=["query", "docs", "k"],
        template="Classify these documents into distinct opinions on '{query}' and select the {k} most representative and diverse viewpoints:\nDocuments: {docs}\nSelected indices:"

    )
    opinion_chain = opinion_prompt | self.llm.with_structured_output(SelectedIndices)

    docs_text = "\n".join([f"{i}: {doc.page_content[:100]}..." for i, doc in enumerate(all_docs)])
    input_data = {"query": query, "docs": docs_text, "k": k}
    selected_indices = opinion_chain.invoke(input_data).indices
    print(f'selected diverse and relevant documents')

    return [all_docs[int(i)] for i in selected_indices.split() if i.isdigit() and int(i) < len(all_docs)]



### Analytical reriever strategy

In [ ]:
class SelectedIndices(BaseModel):
    indices: List[int] = Field(description="Indices of selected documents", example=[0, 1, 2, 3])

class SubQueries(BaseModel):
    sub_queries: List[str] = Field(description="List of sub-queries for comprehensive analysis", example=["What is the population of New York?", "What is the GDP of New York?"])

class AnalyticalRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=4):
        print("retrieving analytical")
        # Use LLM to generate sub-queries for comprehensive analysis
        sub_queries_prompt = PromptTemplate(
            input_variables=["query", "k"],
            template="Generate {k} sub-questions for: {query}"
        )

        llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)
        sub_queries_chain = sub_queries_prompt | llm.with_structured_output(SubQueries)

        input_data = {"query": query, "k": k}
        sub_queries = sub_queries_chain.invoke(input_data).sub_queries
        print(f'sub queries for comprehensive analysis: {sub_queries}')

        all_docs = []
        for sub_query in sub_queries:
            all_docs.extend(self.db.similarity_search(sub_query, k=2))

        # Use LLM to ensure diversity and relevance
        diversity_prompt = PromptTemplate(
            input_variables=["query", "docs", "k"],
            template="""Select the most diverse and relevant set of {k} documents for the query: '{query}'\nDocuments: {docs}\n
            Return only the indices of selected documents as a list of integers."""
        )
        diversity_chain = diversity_prompt | self.llm.with_structured_output(SelectedIndices)
        docs_text = "\n".join([f"{i}: {doc.page_content[:50]}..." for i, doc in enumerate(all_docs)])
        input_data = {"query": query, "docs": docs_text, "k": k}
        selected_indices_result = diversity_chain.invoke(input_data).indices
        print(f'selected diverse and relevant documents')

        return [all_docs[i] for i in selected_indices_result if i < len(all_docs)

### Contextual retriever strategy

In [ ]:

class ContextualRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=4, user_context=None):
        print("retrieving contextual")
        # Use LLM to incorporate user context into the query
        context_prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="Given the user context: {context}\nReformulate the query to best address the user's needs: {query}"
        )
        context_chain = context_prompt | self.llm
        input_data = {"query": query, "context": user_context or "No specific context provided"}
        contextualized_query = context_chain.invoke(input_data).content
        print(f'contextualized query: {contextualized_query}')

        # Retrieve documents using the contextualized query
        docs = self.db.similarity_search(contextualized_query, k=k*2)

        # Use LLM to rank the relevance of retrieved documents considering the user context
        ranking_prompt = PromptTemplate(
            input_variables=["query", "context", "doc"],
            template="Given the query: '{query}' and user context: '{context}', rate the relevance of this document on a scale of 1-10:\nDocument: {doc}\nRelevance score:"
        )
        ranking_chain = ranking_prompt | self.llm.with_structured_output(relevant_score)
        print("ranking docs")

        ranked_docs = []
        for doc in docs:
            input_data = {"query": contextualized_query, "context": user_context or "No specific context provided", "doc": doc.page_content}
            score = float(ranking_chain.invoke(input_data).score)
            ranked_docs.append((doc, score))


        # Sort by relevance score and return top k
        ranked_docs.sort(key=lambda x: x[1], reverse=True)

        return [doc for doc, _ in ranked_docs[:k]]

### Adapive retriever class

In [ ]:
class AdaptiveRetriever:
    def __init__(self, texts: List[str]):
        self.classifier = QueryClassifier()
        self.strategies = {
            "Factual": FactualRetrievalStrategy(texts),
            "Analytical": AnalyticalRetrievalStrategy(texts),
            "Opinion": OpinionRetrievalStrategy(texts),
            "Contextual": ContextualRetrievalStrategy(texts)
        }

    def get_relevant_documents(self, query: str) -> List[Document]:
        category = self.classifier.classify(query)
        strategy = self.strategies[category]
        return strategy.retrieve(query)

### Additional retriever that inherits from langchain BaseRetriever

In [ ]:
class PydanticAdaptiveRetriever(BaseRetriever):
    adaptive_retriever: AdaptiveRetriever = Field(exclude=True)

    class Config:
        arbitrary_types_allowed = True

    def get_relevant_documents(self, query: str) -> List[Document]:
        return self.adaptive_retriever.get_relevant_documents(query)

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        return self.get_relevant_documents(query)